<a href="https://colab.research.google.com/github/kendall-lucas25/women_endurance/blob/main/DTSC_220_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

At what distance, if any, do women outperform men in extreme endurance running?


We are trying to predict the value where this happens

import packages

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

set output options

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

read in dataframe

In [ ]:
races_df = pd.read_csv('TWO_CENTURIES_OF_UM_RACES.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'TWO_CENTURIES_OF_UM_RACES.csv'

get a deeper look on the dataset

In [ ]:
races_df.head()

In [ ]:
races_df.sample(n = 5)

Each observation is an each independent person's finish time. One thing we have to remember is that this entails that if two observations of the same person are in the dataset, then the each individual observations in the dataset are not independent.

In [ ]:
races_df.shape

In [ ]:
races_df.info()

ok this tells the data types of all the variables (only 12 that's not a problem right?)

0(Year of event) --> no nans, integer, check the range of the years

1(Event dates) --> 1 null?, object type so that is probably going to need adjusted

2(Event name) --> 1 null?, object type change to string?

3(Event distance/length) --> decent # of nans, object type so change it to numeric, this is a KEY variable

4(Event number of finishers) --> 1 null?, float?? (how can a fraction of a person finishe???)

5(Athlete Performance) --> 1 null?, is this always mean finish time??, if so we gotta change the type to something else

6(Athlete Club) --> lots of nans but id assume we arent going to be using this variable a lot

7(Athlete Country) --> 1 null, probably not going to be used a ton, we could change the type tho

8(Athlete Year of birth) --> woah a decent number of nans, not sure if we need to fix these or just disregard, we could also engineeer an age column and make this a float/integer (probably float)

9(Athlete Gender) --> Ding ding ding, 1 null, change from object to boolean?

10(Athlete Age Category) --> this varies by race so i dont think im going to use it much

11(Athlete Average Speed) --> this is just distance divided by time so like i wont use it much

12(Athlete ID) --> why is it a float what?, 1 null?, pretty important

1) what is the range of the years in the dataset

In [ ]:
races_df['Year of event'].max()

In [ ]:
races_df['Year of event'].min()

1) where is the year equal to 2??

In [ ]:
races_df.loc[(races_df['Year of event'] == 2)]

Row 389014 is a row full of nans and wrong numbers so im going to delete it

In [ ]:
races_df = races_df.drop(389014)

In [ ]:
races_df['Year of event'].max()

In [ ]:
races_df['Year of event'].min()

ok so the website on kaggle says that this has data from a ton of different years so im going to investigate this missing data

it says dragging and dropping can truncate some data so i am going to load the data in using the kaggle api

In [3]:
#this code is from the kaggle website
import kagglehub

# Download latest version
path = kagglehub.dataset_download("aiaiaidavid/the-big-dataset-of-ultra-marathon-running")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'the-big-dataset-of-ultra-marathon-running' dataset.
Path to dataset files: /kaggle/input/the-big-dataset-of-ultra-marathon-running


In [4]:
#this code is AI generated from google gemini after websearch
import os

# List all files in the downloaded directory
files = os.listdir(path)
print("Files in dataset:", files)

Files in dataset: ['TWO_CENTURIES_OF_UM_RACES', 'TWO_CENTURIES_OF_UM_RACES.csv']


In [5]:
#this code is AI generated from google gemini after websearch
csv_file = os.path.join(path, files[0])

df = pd.read_csv(csv_file)

/tmp/ipykernel_1510/3675276953.py:4: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_file)


In [ ]:
df.head()

,Year of event,Event dates,Event name,Event distance/length,Event number of finishers,Athlete performance,Athlete club,Athlete country,Athlete year of birth,Athlete gender,Athlete age category,Athlete average speed,Athlete ID
0,2018,06.01.2018,Selva Costera (CHI),50km,22,4:51:39 h,Tnfrc,CHI,1978.0,M,M35,10.286,0
1,2018,06.01.2018,Selva Costera (CHI),50km,22,5:15:45 h,Roberto Echeverría,CHI,1981.0,M,M35,9.501,1
2,2018,06.01.2018,Selva Costera (CHI),50km,22,5:16:44 h,Puro Trail Osorno,CHI,1987.0,M,M23,9.472,2
3,2018,06.01.2018,Selva Costera (CHI),50km,22,5:34:13 h,Columbia,ARG,1976.0,M,M40,8.976,3
4,2018,06.01.2018,Selva Costera (CHI),50km,22,5:54:14 h,Baguales Trail,CHI,1992.0,M,M23,8.469,4


In [ ]:
df.shape

(7117634, 13)

this is the correct number!!!

In [ ]:
df.describe()

,Year of event,Event number of finishers,Athlete year of birth,Athlete ID
count,7.117634e+06,7.117634e+06,6.558280e+06,7.117634e+06
mean,2.012990e+03,1.310716e+03,1.970613e+03,5.275223e+05
std,9.074683e+00,2.973447e+03,1.264591e+01,4.634639e+05
min,1.798000e+03,0.000000e+00,1.193000e+03,0.000000e+00
25%,2.011000e+03,8.700000e+01,1.963000e+03,1.304470e+05
50%,2.016000e+03,2.270000e+02,1.972000e+03,3.775990e+05
75%,2.019000e+03,7.690000e+02,1.980000e+03,9.164148e+05
max,2.022000e+03,2.002700e+04,2.021000e+03,1.600755e+06


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7117634 entries, 0 to 7117633
Data columns (total 13 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   Year of event              int64  
 1   Event dates                object 
 2   Event name                 object 
 3   Event distance/length      object 
 4   Event number of finishers  int64  
 5   Athlete performance        object 
 6   Athlete club               object 
 7   Athlete country            object 
 8   Athlete year of birth      float64
 9   Athlete gender             object 
 10  Athlete age category       object 
 11  Athlete average speed      object 
 12  Athlete ID                 int64  
dtypes: float64(1), int64(3), object(9)
memory usage: 705.9+ MB


this is the same as above so I'm going to continue with data cleaning with the new dataset df

0. what is the range of the years in this dataset

In [ ]:
df['Year of event'].max()

2022

In [ ]:
df['Year of event'].min()

1798

we have data from 1798 that's crazy (great)

0. check for nans in year column

In [ ]:
df['Year of event'].isna().sum()

np.int64(0)

yay no nans!

1. check for nans in the event date column

In [ ]:
df['Event dates'].isna().sum()

np.int64(0)

yay no nans!

Now going to clean data in order of importance to the project

3(Event Distance) is really important

In [ ]:
df.loc[df['Event distance/length'].isna(), 'Event name'].unique()

array(['Maratonmarschen Stockholm (SWE)', 'Stockholm Fotrally (SWE)'],
      dtype=object)

In [ ]:
df.loc[(df['Event name'] == 'Maratonmarschen Stockholm (SWE)') | (df['Event name'] == 'Stockholm Fotrally (SWE)')]

,Year of event,Event dates,Event name,Event distance/length,Event number of finishers,Athlete performance,Athlete club,Athlete country,Athlete year of birth,Athlete gender,Athlete age category,Athlete average speed,Athlete ID,Start date
380797,2018,29.06.-03.07.2018,Maratonmarschen Stockholm (SWE),NaN,174,3d 15:48:00 km,*Lund,SWE,1967.0,M,M50,0,88295,29.06
380798,2018,29.06.-03.07.2018,Maratonmarschen Stockholm (SWE),NaN,174,3d 15:48:00 km,*Södertälje,SWE,1978.0,F,W40,0,249664,29.06
380799,2018,29.06.-03.07.2018,Maratonmarschen Stockholm (SWE),NaN,174,2d 05:08:00 km,*Lidingö,SWE,1963.0,M,M55,0,249665,29.06
380800,2018,29.06.-03.07.2018,Maratonmarschen Stockholm (SWE),NaN,174,2d 05:00:00 km,*Örebro,SWE,1978.0,M,M35,0,249666,29.06
380801,2018,29.06.-03.07.2018,Maratonmarschen Stockholm (SWE),NaN,174,2d 02:35:00 km,*Täby,SWE,1972.0,F,W45,0,110881,29.06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6912620,2015,25.-29.06.2015,Stockholm Fotrally (SWE),NaN,143,8:45:00 km,*Åkersberga,SWE,NaN,M,NaN,0.0,1579046,25
6912621,2015,25.-29.06.2015,Stockholm Fotrally (SWE),NaN,143,8:40:00 km,*Åkersberga,SWE,1990.0,M,M23,0.0,249710,25
6912622,2015,25.-29.06.2015,Stockholm Fotrally (SWE),NaN,143,8:30:00 km,*Rosersberg,SWE,1979.0,F,W35,0.0,114555,25
6912623,2015,25.-29.06.2015,Stockholm Fotrally (SWE),NaN,143,8:03:00 km,*Stockholm,SWE,NaN,M,NaN,0.0,1579047,25


In [ ]:
df.loc[df['Event distance/length'].isna()]

,Year of event,Event dates,Event name,Event distance/length,Event number of finishers,Athlete performance,Athlete club,Athlete country,Athlete year of birth,Athlete gender,Athlete age category,Athlete average speed,Athlete ID,Start date
380797,2018,29.06.-03.07.2018,Maratonmarschen Stockholm (SWE),NaN,174,3d 15:48:00 km,*Lund,SWE,1967.0,M,M50,0,88295,29.06
380798,2018,29.06.-03.07.2018,Maratonmarschen Stockholm (SWE),NaN,174,3d 15:48:00 km,*Södertälje,SWE,1978.0,F,W40,0,249664,29.06
380799,2018,29.06.-03.07.2018,Maratonmarschen Stockholm (SWE),NaN,174,2d 05:08:00 km,*Lidingö,SWE,1963.0,M,M55,0,249665,29.06
380800,2018,29.06.-03.07.2018,Maratonmarschen Stockholm (SWE),NaN,174,2d 05:00:00 km,*Örebro,SWE,1978.0,M,M35,0,249666,29.06
380801,2018,29.06.-03.07.2018,Maratonmarschen Stockholm (SWE),NaN,174,2d 02:35:00 km,*Täby,SWE,1972.0,F,W45,0,110881,29.06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6912620,2015,25.-29.06.2015,Stockholm Fotrally (SWE),NaN,143,8:45:00 km,*Åkersberga,SWE,NaN,M,NaN,0.0,1579046,25
6912621,2015,25.-29.06.2015,Stockholm Fotrally (SWE),NaN,143,8:40:00 km,*Åkersberga,SWE,1990.0,M,M23,0.0,249710,25
6912622,2015,25.-29.06.2015,Stockholm Fotrally (SWE),NaN,143,8:30:00 km,*Rosersberg,SWE,1979.0,F,W35,0.0,114555,25
6912623,2015,25.-29.06.2015,Stockholm Fotrally (SWE),NaN,143,8:03:00 km,*Stockholm,SWE,NaN,M,NaN,0.0,1579047,25


In [ ]:
df = df.dropna(subset = ['Event distance/length'])

In [ ]:
df.shape

(7116581, 14)

All nans in event distnace/length are taken care of

only keep distance/length to the actual distance no times (the concept of the race is distance not how long can someone last)

In [ ]:
df.head()

,Year of event,Event dates,Event name,Event distance/length,Event number of finishers,Athlete performance,Athlete club,Athlete country,Athlete year of birth,Athlete gender,Athlete age category,Athlete average speed,Athlete ID
0,2018,06.01.2018,Selva Costera (CHI),50km,22,4:51:39 h,Tnfrc,CHI,1978.0,M,M35,10.286,0
1,2018,06.01.2018,Selva Costera (CHI),50km,22,5:15:45 h,Roberto Echeverría,CHI,1981.0,M,M35,9.501,1
2,2018,06.01.2018,Selva Costera (CHI),50km,22,5:16:44 h,Puro Trail Osorno,CHI,1987.0,M,M23,9.472,2
3,2018,06.01.2018,Selva Costera (CHI),50km,22,5:34:13 h,Columbia,ARG,1976.0,M,M40,8.976,3
4,2018,06.01.2018,Selva Costera (CHI),50km,22,5:54:14 h,Baguales Trail,CHI,1992.0,M,M23,8.469,4


In [ ]:
df['Event distance/length'].describe()

,Event distance/length
count,7116581
unique,2131
top,50km
freq,1503475


In [ ]:
df['Event distance/length'].value_counts()

,count
Event distance/length,
50km,1503475
100km,883268
50mi,333685
56km,274234
24h,172811
100mi,164750
87km,162087
60km,156806
45km,155475


In [ ]:
df['Event distance/length'].str.contains('km').sum()

np.int64(5862259)

In [ ]:
df['Event distance/length'].str.contains('mi').sum()

np.int64(742583)

In [6]:
df = df.loc[~((df['Event distance/length'].str.contains('/')) | (df['Event distance/length'].str.contains(',')) | (df['Event distance/length'].str.contains('Split')))]

The code above gets rid of rows where the dataset has / or , in the race distance column, which is when the race has stages or loops and isn't just a full race

In [ ]:
df['Event distance/length'].value_counts()

,count
Event distance/length,
50km,1503475
100km,883268
50mi,333685
56km,274234
24h,172811
...,...
105.8mi,1
431km,1
27:30h,1


In [7]:
df = df[df['Event distance/length'].str.contains('km') | df['Event distance/length'].str.contains('mi')]

The code above remove rows from the dataset where the goal of the race is to last a length of time, and not a length of distance (only keep races measured in km or mi)

In [ ]:
df['Event distance/length'].value_counts()

,count
Event distance/length,
50km,1503475
100km,883268
50mi,333685
56km,274234
100mi,164750
...,...
303mi,1
186mi,1
431km,1


In [ ]:
df.shape

(6513668, 13)

ok this structure only keeps 91% of the dataset, so we still have a lot of our original data.

If i've done everything correctly then the athlete performance should have no distance result

In [ ]:
df[df['Athlete performance'].str.contains('km') | df['Athlete performance'].str.contains('mi')]

,Year of event,Event dates,Event name,Event distance/length,Event number of finishers,Athlete performance,Athlete club,Athlete country,Athlete year of birth,Athlete gender,Athlete age category,Athlete average speed,Athlete ID


ok nice

The three main columns i have to look at are Event distance/length, Athlete Performance, and Athlete gender (maybe the year of the race to just to make sure the model isnt training on loads of data from the 19th century)

Going to continue cleaning the event distance/length column to put everything into miles (then maybe change the title of the column to event distance because we got rid of all the results that have an event length)

In [8]:
def to_miles(col):
  if 'km' in col:
    return float(col.split('km')[0])*0.621
  else:
    return float(col.split('mi')[0])

In [9]:
df = df.loc[~(df['Event distance/length'].str.contains('4x52'))]

In [10]:
df['miles'] = df['Event distance/length'].apply(to_miles)

In [ ]:
df['Event distance/length'].value_counts()

,count
Event distance/length,
50km,1522609
100km,951742
50mi,352181
56km,333253
87km,212062
...,...
105.8mi,1
505km,1
158.5mi,1


In [ ]:
df['miles'].value_counts()

,count
miles,
31.05000,1523076
62.10000,951762
50.00000,352345
34.77600,333303
54.02700,212135
...,...
158.50000,1
37.97415,1
178.84800,1


these values dont match up going to look into that

In [20]:
mi = df.loc[df['Event distance/length'].str.contains('mi')]

In [24]:
km = df.loc[df['Event distance/length'].str.contains('km')]
km.loc[((km['Event distance/length'].str.split('km').str[0].astype(float) * 0.621) != km['miles'])]

,Year of event,Event dates,Event name,Event distance/length,Event number of finishers,Athlete performance,Athlete club,Athlete country,Athlete year of birth,Athlete gender,Athlete age category,Athlete average speed,Athlete ID,miles


In [26]:
mi.loc[((mi['Event distance/length'].str.split('mi').str[0].astype(float)) != mi['miles'])]

,Year of event,Event dates,Event name,Event distance/length,Event number of finishers,Athlete performance,Athlete club,Athlete country,Athlete year of birth,Athlete gender,Athlete age category,Athlete average speed,Athlete ID,miles


ok so it seems that the function works correctly

In [28]:
df['miles'].dtype

dtype('float64')

Event distance/length is all cleaned up (besides column title), so now we can move onto athlete performance

ok wait now im just going to look at races. that hvae women in them